# Milestone3

### Notebook initialization 

In [100]:
import time
import sys
import requests
import numpy as np
import pandas as pd
import builtins
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import sqrt, pow, round
from pyspark.ml.regression import DecisionTreeRegressor
from xgboost.spark import SparkXGBRegressor
from pyspark import StorageLevel

In [2]:
# Define the library path
lib_path = "/expanse/lustre/scratch/apasvenskas/temp_project/python_libs"

# Ensure the driver has it
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

# Build the new session with executorEnv.PYTHONPATH
# Updated for 32 cores and 150GB of RAM
spark = (SparkSession.builder
    .appName("Waymo_XGBoost_Evaluation_Safe")
    .config("spark.driver.memory", "8g")       # INCREASED from 2g to 8g
    .config("spark.executor.instances", 7)     
    .config("spark.executor.cores", 4)         
    .config("spark.executor.memory", "16g")    # Adjusted to 16g to keep total under 150GB
    .config("spark.executor.memoryOverhead", "2g") 
    .config("spark.executorEnv.PYTHONPATH", lib_path)
    .getOrCreate())

print("Fresh Spark session built with native worker paths!")

Fresh Spark session built with native worker paths!


### Load data

In [3]:
print("Loading data...")
start_time = time.time()
df = spark.read.parquet("/expanse/lustre/scratch/apasvenskas/temp_project/waymo_features.parquet")

print(f"Data loaded in {time.time() - start_time:.2f} seconds.")

Loading data...
Data loaded in 4.66 seconds.


********************
## A data subset for initial testing. 

In [4]:
print("Full rows for Milestone3")
test_df = df
test_df

Full rows for Milestone3


DataFrame[scenario_id: string, track_id: bigint, past_x_0: double, past_y_0: double, past_x_1: double, past_y_1: double, past_x_2: double, past_y_2: double, past_x_3: double, past_y_3: double, past_x_4: double, past_y_4: double, past_x_5: double, past_y_5: double, past_x_6: double, past_y_6: double, past_x_7: double, past_y_7: double, past_x_8: double, past_y_8: double, past_x_9: double, past_y_9: double, past_x_10: double, past_y_10: double, future_x: array<double>, future_y: array<double>]

### Pack features and and historical data into single column.

In [5]:
pipeline_start = time.time()

In [6]:
# Feature Engineering 

exprs = [col("*")]

# Exisitng velocity calc
exprs.append((col("past_x_10") - col("past_x_0")).alias("v_x"))
exprs.append((col("past_y_10") - col("past_y_0")).alias("v_y"))

# Existing delta calc
exprs.append((col("future_x").getItem(9) - col("past_x_10")).alias("target_dx_1s"))
exprs.append((col("future_y").getItem(9) - col("past_y_10")).alias("target_dy_1s"))

for i in range(11):
    exprs.append((col(f"past_x_{i}") - col("past_x_0")).alias(f"rel_x_{i}"))
    exprs.append((col(f"past_y_{i}") - col("past_y_0")).alias(f"rel_y_{i}"))

prep_df = test_df.select(*exprs)

#### Clean Data

In [7]:
clean_df = prep_df.filter(
    (col("target_dx_1s") >= -40) & (col("target_dx_1s") <= 40) &
    (col("target_dy_1s") >= -40) & (col("target_dy_1s") <= 40)
)

In [8]:
feature_cols = (
    [f"rel_x_{i}" for i in range(11)] + 
    [f"rel_y_{i}" for i in range(11)] + 
    ["v_x", "v_y"]
)

In [9]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features") 

In [10]:
scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", 
                        withStd=True, withMean=True)

In [11]:
pipeline = Pipeline(stages=[assembler, scaler])
ml_final = pipeline.fit(clean_df).transform(clean_df)
ml_ready_df = ml_final.select("scenario_id", "track_id", "scaled_features", "target_dx_1s", "target_dy_1s")

### Train/Split the data set 80/20, and train the GBTRegressor

In [12]:
train_df, eval_df = ml_ready_df.randomSplit([0.8, 0.2], seed=42)

#### Architectural Note:

Model Selection & Memory Constraints:
********

The initial architecture for this pipeline utilized PySpark's native GBTRegressor. However, scaling this model to the full 30GB Waymo dataset caused catastrophic Out-Of-Memory (OOM) failures on the JVM. The model consistently crashed the cluster despite utilizing a heavy distributed configuration (7 executors, 4 cores each, 15GB memory per executor, plus 2GB overhead) on a 130GB+ compute node.

Because the native implementation could not construct the required gradient histograms within a 150GB memory footprint, the pipeline was transitioned to SparkXGBRegressor. By leveraging XGBoost's highly optimized C++ backend, the model was able to manage memory much more efficiently, successfully completing the training phase well within the cluster's hardware limits.

In [13]:
print("Training Distributed XGBoost on the FULL 30GB dataset")
start_xgb = time.time()

Training Distributed XGBoost on the FULL 30GB dataset


In [14]:
xgb_x = SparkXGBRegressor(
    features_col="scaled_features", 
    label_col="target_dx_1s",
    num_workers=7,          
    max_depth=5,            
    n_estimators=20,        
    use_gpu=False           
)

In [96]:
xgb_model_x = xgb_x.fit(train_df)
print(f"Full Dataset Training successful in {time.time() - start_xgb:.2f} seconds!")

2026-05-13 12:27:05,854 INFO XGBoost-PySpark: _fit Running xgboost-2.0.3 on 7 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'max_depth': 5, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 20}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-05-13 12:27:11,755 INFO XGBoost-PySpark: _fit Finished xgboost training!


Full Dataset Training successful in 2346.12 seconds!


In [16]:
print("Generating and safely caching predictions...")
pred_xgb_x = xgb_model_x.transform(eval_df)
pred_xgb_x.persist(StorageLevel.MEMORY_AND_DISK)

Generating and safely caching predictions...


DataFrame[scenario_id: string, track_id: bigint, scaled_features: vector, target_dx_1s: double, target_dy_1s: double, prediction: double]

In [17]:
# Force the predictions to materialize NOW, before the evaluator runs
pred_count = pred_xgb_x.count()
print(f"Materialized {pred_count} predictions safely into memory.")

Materialized 154714 predictions safely into memory.


In [18]:
print("Calculating final RMSE.")
evaluator = RegressionEvaluator(labelCol="target_dx_1s", predictionCol="prediction", metricName="rmse")
rmse_xgb_x = evaluator.evaluate(pred_xgb_x)

Calculating final RMSE.


In [19]:
print(f"FULL DATASET RMSE for X: {rmse_xgb_x:.4f} meters.")

FULL DATASET RMSE for X: 0.4746 meters.


In [20]:
print("XGBoost: Actual vs Predicted")
pred_xgb_x.select("track_id", "target_dx_1s", "prediction").show(5, truncate=False)

XGBoost: Actual vs Predicted
+--------+------------+---------------------+
|track_id|target_dx_1s|prediction           |
+--------+------------+---------------------+
|322     |0.0         |-6.257289205677807E-4|
|329     |0.0         |-6.257289205677807E-4|
|333     |9.765625E-4 |-6.257289205677807E-4|
|338     |0.0         |-6.257289205677807E-4|
|345     |0.0         |-6.257289205677807E-4|
+--------+------------+---------------------+
only showing top 5 rows



**************
## Train Y cordinate. 

In [21]:
xgb_y = SparkXGBRegressor(
    features_col="scaled_features", 
    label_col="target_dy_1s",       # <-- Change this to Y
    num_workers=7,          
    max_depth=5,            
    n_estimators=20,        
    use_gpu=False           
)

In [22]:
start_xgb = time.time()
xgb_model_y = xgb_y.fit(train_df)
print(f"Y-Model Training successful in {time.time() - start_xgb:.2f} seconds!")

2026-05-13 11:48:05,783 INFO XGBoost-PySpark: _fit Running xgboost-2.0.3 on 7 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'max_depth': 5, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 20}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-05-13 11:48:12,630 INFO XGBoost-PySpark: _fit Finished xgboost training!


Y-Model Training successful in 7.00 seconds!


#### Evaluation

In [23]:
pred_xgb_y = xgb_model_y.transform(eval_df)
pred_xgb_y.persist(StorageLevel.MEMORY_AND_DISK)
pred_count_y = pred_xgb_y.count()

In [24]:
evaluator_y = RegressionEvaluator(labelCol="target_dy_1s", predictionCol="prediction", metricName="rmse")
rmse_xgb_y = evaluator_y.evaluate(pred_xgb_y)

In [25]:
print(f"FULL DATASET RMSE for Y: {rmse_xgb_y:.4f} meters.")

FULL DATASET RMSE for Y: 0.4893 meters.


In [26]:
print("XGBoost: Actual vs Predicted")
pred_xgb_y.select("track_id", "target_dy_1s", "prediction").show(5, truncate=False)

XGBoost: Actual vs Predicted
+--------+-------------+---------------------+
|track_id|target_dy_1s |prediction           |
+--------+-------------+---------------------+
|322     |0.0          |-9.249793947674334E-4|
|329     |0.0          |-9.249793947674334E-4|
|333     |0.00244140625|-9.249793947674334E-4|
|338     |0.0          |-9.249793947674334E-4|
|345     |0.0          |-9.249793947674334E-4|
+--------+-------------+---------------------+
only showing top 5 rows



**********
## Trajectory Combiner

In [27]:
pred_x_df = xgb_model_x.transform(eval_df).withColumnRenamed("prediction", "pred_dx_1s")
final_trajectory_df = xgb_model_y.transform(pred_x_df).withColumnRenamed("prediction", "pred_dy_1s")

In [28]:
final_trajectory_df = final_trajectory_df.withColumn(
    "spatial_error_meters",
    sqrt(pow(col("target_dx_1s") - col("pred_dx_1s"), 2) + 
         pow(col("target_dy_1s") - col("pred_dy_1s"), 2))
)

In [29]:
print("Final Combined Trajectories: Actual vs Predicted")
final_trajectory_df.select(
    "track_id", 
    "target_dx_1s", "pred_dx_1s", 
    "target_dy_1s", "pred_dy_1s",
    "spatial_error_meters"
).show(10, truncate=False)

Final Combined Trajectories: Actual vs Predicted
+--------+--------------------+---------------------+--------------------+---------------------+---------------------+
|track_id|target_dx_1s        |pred_dx_1s           |target_dy_1s        |pred_dy_1s           |spatial_error_meters |
+--------+--------------------+---------------------+--------------------+---------------------+---------------------+
|322     |0.0                 |-6.257289205677807E-4|0.0                 |-9.249793947674334E-4|0.0011167468660261588|
|329     |0.0                 |-6.257289205677807E-4|0.0                 |-9.249793947674334E-4|0.0011167468660261588|
|333     |9.765625E-4         |-6.257289205677807E-4|0.00244140625       |-9.249793947674334E-4|0.0037282556384616874|
|338     |0.0                 |-6.257289205677807E-4|0.0                 |-9.249793947674334E-4|0.0011167468660261588|
|345     |0.0                 |-6.257289205677807E-4|0.0                 |-9.249793947674334E-4|0.0011167468660261588|

**********
## Compare training vs test error.

In [30]:
# Evaluate training X.
evaluator_x = RegressionEvaluator(labelCol="target_dx_1s", predictionCol="prediction", metricName="rmse")
pred_train_x = xgb_model_x.transform(train_df)
rmse_train_x = evaluator_x.evaluate(pred_train_x)

In [31]:
# 2. Evaluate Training Y
evaluator_y = RegressionEvaluator(labelCol="target_dy_1s", predictionCol="prediction", metricName="rmse")
pred_train_y = xgb_model_y.transform(train_df)
rmse_train_y = evaluator_y.evaluate(pred_train_y)

In [32]:
print(f"X-Coordinate | Training RMSE: {rmse_train_x:.4f} m | Test RMSE: {rmse_xgb_x:.4f} m")
print(f"Y-Coordinate | Training RMSE: {rmse_train_y:.4f} m | Test RMSE: {rmse_xgb_y:.4f} m")

X-Coordinate | Training RMSE: 0.4649 m | Test RMSE: 0.4746 m
Y-Coordinate | Training RMSE: 0.4891 m | Test RMSE: 0.4893 m


In [33]:
print("Training Data Sample: Actual vs Predicted (X-Coordinate)")
pred_train_x.select("track_id", "target_dx_1s", "prediction").show(5, truncate=False)

Training Data Sample: Actual vs Predicted (X-Coordinate)
+--------+------------+---------------------+
|track_id|target_dx_1s|prediction           |
+--------+------------+---------------------+
|318     |0.3466796875|0.38126805424690247  |
|320     |0.0         |-6.257289205677807E-4|
|323     |0.0         |-6.257289205677807E-4|
|326     |0.0         |-6.257289205677807E-4|
|328     |9.765625E-4 |-0.017257947474718094|
+--------+------------+---------------------+
only showing top 5 rows



******
## Model with different Hyperparameters. 

#### Hyperparameter Tunning

In [34]:
print("Training DEEP XGBoost Model (Hyperparameter Tuning) on FULL 30GB dataset.")
start_deep_xgb = time.time()

Training DEEP XGBoost Model (Hyperparameter Tuning) on FULL 30GB dataset.


#### Hyperparameter Tunning

In [35]:
xgb_deep = SparkXGBRegressor(
    features_col="scaled_features", 
    label_col="target_dx_1s",
    num_workers=7,          
    max_depth=10,           # DEEPER TREES: Captures more complex physics
    n_estimators=40,        # MORE TREES: Stronger ensemble learning
    use_gpu=False           
)

#### Train a new model.

In [36]:
xgb_model_deep = xgb_deep.fit(train_df)
print(f"Deep Model Training successful in {time.time() - start_deep_xgb:.2f} seconds!")

2026-05-13 11:48:22,075 INFO XGBoost-PySpark: _fit Running xgboost-2.0.3 on 7 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'max_depth': 10, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 40}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-05-13 11:48:30,072 INFO XGBoost-PySpark: _fit Finished xgboost training!


Deep Model Training successful in 8.24 seconds!


#### Evaluate the new model.

In [37]:
print("Generating Deep Model Predictions.")
pred_deep_x = xgb_model_deep.transform(eval_df)
pred_deep_x.persist(StorageLevel.MEMORY_AND_DISK)
pred_deep_x.count() 

Generating Deep Model Predictions.


154714

In [109]:
evaluator_deep = RegressionEvaluator(labelCol="target_dx_1s", predictionCol="prediction", metricName="rmse")
rmse_deep_x = evaluator_deep.evaluate(pred_deep_x)

In [112]:
print("Evaluating Deep Model on Training Data to check for overfitting.")
pred_deep_train_x = xgb_model_deep.transform(train_df)
rmse_deep_train_x = evaluator_deep.evaluate(pred_deep_train_x)

Evaluating Deep Model on Training Data to check for overfitting.


In [111]:
print(f"Deep Tree Model | Training RMSE: {rmse_deep_train_x:.4f} m | Test RMSE: {rmse_deep_x:.4f} m\n")
print(f"Baseline (Depth 5) Test RMSE:  {rmse_xgb_x:.4f} meters")
print(f"Deep Tree (Depth 10) Test RMSE: {rmse_deep_x:.4f} meters")

Deep Tree Model | Training RMSE: 0.3474 m | Test RMSE: 0.4274 m

Baseline (Depth 5) Test RMSE:  0.4746 meters
Deep Tree (Depth 10) Test RMSE: 0.4274 meters


*******************
## Spark session data Executor Summary, Stage Metrics (Shuffle data), Data Skew (Max/Median task ratio)

#### API connection setup.

In [40]:
base_url = "http://localhost:4040/api/v1/applications"
app_info = requests.get(base_url).json()
app_id = app_info[0]['id']
app_url = f"{base_url}/{app_id}"

#### Proof of cluster configuration

In [107]:
print("1. THE 'LOCAL MODE' REVELATION")
print("         ")

master_config = spark.sparkContext.master
print(f"Spark Master Mode: {master_config}")
print("Because the allocation is on a single Expanse node, Spark combines all")
print("resources into a single super-powered 'driver' executor.")

print("         ")
print("2. PROOF OF DISTRIBUTED COMPUTE CORES")

base_url = "http://localhost:4040/api/v1/applications"
app_info = requests.get(base_url).json()
app_id = app_info[0]['id']

executors = requests.get(f"{base_url}/{app_id}/executors").json()

exec_data = [{
    "Executor Name": e.get('id').upper(), 
    "Status": "Active & Executing",
    "Total Cores Utilized": e.get('totalTasks', e.get('totalCores', 'Max')), 
    # FIX: Using builtins.round() instead of the PySpark round()
    "Assigned Memory (GB)": builtins.round(e.get('maxMemory', 0) / (1024**3), 2)
} for e in executors]

display(pd.DataFrame(exec_data))

1. THE 'LOCAL MODE' REVELATION
         
Spark Master Mode: local[*]
Because the allocation is on a single Expanse node, Spark combines all
resources into a single super-powered 'driver' executor.
         
2. PROOF OF DISTRIBUTED COMPUTE CORES


,Executor Name,Status,Total Cores Utilized,Assigned Memory (GB)
0,DRIVER,Active & Executing,2536,4.62


#### Shuffle data

In [62]:
stages = requests.get(f"{app_url}/stages").json()

In [63]:
stages_sorted = sorted(stages, key=lambda x: x.get('executorRunTime', 0), reverse=True)
heaviest_stage = stages_sorted[0]
stage_id = heaviest_stage['stageId']
stage_attempt = heaviest_stage['attemptId']

In [64]:
print(f"Target Stage Name: {heaviest_stage['name']}")
print(f"Stage ID: {stage_id}")
print(f"Total Shuffle Read:  {heaviest_stage.get('shuffleReadBytes', 0) / (1024**3):.4f} GB")
print(f"Total Shuffle Write: {heaviest_stage.get('shuffleWriteBytes', 0) / (1024**3):.4f} GB")

Target Stage Name: fit at /scratch/apasvenskas/job_48952043/ipykernel_3493607/12224439.py:1
Stage ID: 6
Total Shuffle Read:  0.0248 GB
Total Shuffle Write: 0.0000 GB


#### Data Skew Analysis (Max/Median)

In [65]:
stage_details = requests.get(f"{app_url}/stages/{stage_id}/{stage_attempt}").json()
tasks = stage_details.get('tasks', {})

In [66]:
durations = [t.get('duration', 0) / 1000 for t in tasks.values() if 'duration' in t]

In [67]:
if durations:
    max_duration = np.max(durations)
    median_duration = np.median(durations)
    ratio = max_duration / median_duration
    
    print(f"Max Task Duration:    {max_duration:.2f} seconds")
    print(f"Median Task Duration: {median_duration:.2f} seconds")
    print(f"Max/Median Ratio:     {ratio:.2f}x")
    
    print("\n--- CONCLUSION ---")
    if ratio < 1.5:
        print("Status: Normal. Workload is perfectly balanced across your 7 executors!")
    elif ratio < 3.0:
        print("Status: Mild Skew. Acceptable for this dataset size.")
    else:
        print("Status: Severe Skew. You might need to use the 'Salting' strategy in the future.")
else:
    print("Could not retrieve task durations. Ensure an action like .count() was run!")

Max Task Duration:    33.23 seconds
Median Task Duration: 33.18 seconds
Max/Median Ratio:     1.00x

--- CONCLUSION ---
Status: Normal. Workload is perfectly balanced across your 7 executors!
